In [2]:
!pip install transformers datasets torch accelerate -q

You should consider upgrading via the 'C:\Users\Azam Charaniya\OneDrive\Desktop\AI_journey\ollama\projects\week1-nlp-foundations\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [4]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)
print(dataset['train'].features)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War an

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch

# Small subset for speed
train_subset = dataset['train'].select(range(500))
test_subset = dataset['test'].select(range(100))

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=256)

train_tokenized = train_subset.map(tokenize, batched=True)
test_tokenized = test_subset.map(tokenize, batched=True)

train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

train_loader = DataLoader(train_tokenized, batch_size=8, shuffle=True)
test_loader = DataLoader(test_tokenized, batch_size=8)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Map: 100%|██████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 1436.58 examples/s]

Train batches: 63
Test batches: 13


In [7]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
optimizer = AdamW(model.parameters(), lr=2e-5)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")
model.to(device)

# One epoch
model.train()
total_loss = 0
for batch in train_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['label'].to(device)

    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

print(f"Average training loss: {total_loss / len(train_loader):.4f}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training on: cpu
Average training loss: 0.0892


In [8]:
from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=['negative', 'positive']))

ValueError: Number of classes, 1, does not match size of target_names, 2. Try specifying the labels parameter

In [9]:
print(classification_report(all_labels, all_preds, target_names=['negative', 'positive'], labels=[0,1], zero_division=0))

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       100
    positive       0.00      0.00      0.00         0

    accuracy                           1.00       100
   macro avg       0.50      0.50      0.50       100
weighted avg       1.00      1.00      1.00       100



Why learning rate is small in fine-tuning
The model is already pretrained with rich linguistic knowledge — large updates would overwrite those weights, destroying what it already knows. A small learning rate (2e-5) nudges the model to specialize without forgetting its foundation.
Insufficient training data — majority class collapse
With too little data and too few epochs, the model doesn't learn enough signal to distinguish classes and defaults to predicting the majority class every time. The result looks like 100% accuracy but is completely useless — exactly the class imbalance failure mode, caused by undertraining instead of skewed data.
Hardware constraints — know when to move to cloud GPU
CPU training caps what you can realistically fine-tune locally — small subsets, few epochs, slow iteration. When local hardware limits your ability to learn, move to Google Colab with a T4 GPU rather than drawing wrong conclusions from an underpowered experiment.